In [ ]:
from src.data_loader import load_split_from_local_files
from src.preprocessing import preprocess_dataset

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import CountVectorizer

In [ ]:
path_to_train = "DialoGPT/sample_data/Train"

train_data = load_split_from_local_files(path_to_train, "train")

print("Raw dialogues:", len(train_data))

In [ ]:
cleaned = preprocess_dataset(train_data)

print("After cleaning:", len(cleaned))

In [ ]:
emotion_map = {
    0: 'no emotion',
    1: 'anger',
    2: 'disgust',
    3: 'fear',
    4: 'happiness',
    5: 'sadness',
    6: 'surprise'
}

In [ ]:
all_emotions = [e for d in cleaned for e in d['emotion']]
labels = [emotion_map[e] for e in all_emotions]

plt.figure(figsize=(10,6))
sns.countplot(y=labels, order=pd.Series(labels).value_counts().index)
plt.title("Emotion Distribution (Cleaned Data)")
plt.xlabel("Count")
plt.ylabel("Emotion")
plt.show()

In [ ]:
utterance_lengths = [
    len(turn.split())
    for d in cleaned
    for turn in d['dialog']
]

print(pd.Series(utterance_lengths).describe())

In [ ]:
plt.figure(figsize=(12,6))
sns.histplot(utterance_lengths, bins=50, kde=True)
plt.title("Utterance Length Distribution")
plt.xlabel("Words per utterance")
plt.ylabel("Frequency")
plt.xlim(0,50)
plt.show()

In [ ]:
text_by_emotion = {label: [] for label in emotion_map.keys()}

for dialogue in cleaned:
    for i, turn in enumerate(dialogue['dialog']):
        emo = dialogue['emotion'][i]
        text_by_emotion[emo].append(turn)

for k in text_by_emotion:
    text_by_emotion[k] = " ".join(text_by_emotion[k])

In [ ]:
def plot_top_ngrams(text, emotion_name, n=1, top_n=15):

    vec = CountVectorizer(ngram_range=(n,n), stop_words='english').fit([text])
    bag = vec.transform([text])
    sum_words = bag.sum(axis=0)

    words_freq = [
        (word, sum_words[0, idx])
        for word, idx in vec.vocabulary_.items()
    ]

    words_freq = sorted(words_freq, key=lambda x: x[1], reverse=True)

    df = pd.DataFrame(words_freq[:top_n], columns=['N-gram', 'Frequency'])

    plt.figure(figsize=(10,6))
    sns.barplot(x='Frequency', y='N-gram', data=df)
    plt.title(f"Top {top_n} {'Bigrams' if n==2 else 'Words'} - {emotion_name}")
    plt.show()

In [ ]:
plot_top_ngrams(text_by_emotion[4], "happiness", n=1)
plot_top_ngrams(text_by_emotion[1], "anger", n=2)
plot_top_ngrams(text_by_emotion[5], "sadness", n=2)

In [ ]:
labels = list(emotion_map.values())

transition_matrix = pd.DataFrame(0, index=labels, columns=labels)

for dialogue in cleaned:
    emotions = dialogue['emotion']

    for i in range(len(emotions)-1):
        from_e = emotion_map[emotions[i]]
        to_e = emotion_map[emotions[i+1]]

        transition_matrix.loc[from_e, to_e] += 1

In [ ]:
plt.figure(figsize=(12,10))
sns.heatmap(
    transition_matrix,
    annot=True,
    fmt='d',
    cmap='viridis'
)
plt.title("Emotion Transition Heatmap")
plt.xlabel("To Emotion")
plt.ylabel("From Emotion")
plt.show()